In [1]:
###!/usr/bin/env python
################################################
# New style 
# ###############################################
import sys

rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")

from Utils import MyConstants as Co

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# for smoothing , nonlienar colors ...
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter
import matplotlib.colors as mcolors

# Some other useful packages 
import copy
import time
import cftime
import yaml
import numbers

# Some other useful packages 
import importlib
from pathlib import Path

grav=Co.grav()

 a path to ../ added in __main__ 
 Utils.MyConstants in /glade/work/juliob/HiRes_ana_dev/Drivers/Utils 


In [2]:
topofile = '/glade/work/juliob/Topo/NCARTopoJTB/cases/fv1x1_Sco100_GrnlAnt/output/fv1x1_gmted2010_modis_bedmachine_nc3000_Laplace0100_noleak_greenlndantarcsgh30fac2.50_20251009.nc'
Topo=xr.open_dataset( topofile )




anglx=Topo.ANGLX.values[0,:,:]
angll=Topo.ANGLL.values[0,:,:]
mxdis=Topo.MXDIS.values[0,:,:]
htopo=Topo.PHIS.values[:,:] /grav
isovar=Topo.ISOVAR.values[:,:]

deg2rad = Co.pi()/180.

angll_rad=angll*deg2rad

sin_angll_rad = np.where( mxdis>1e-6 , np.sin(angll_rad), -9999. )
cos_angll_rad = np.where( mxdis>1e-6 , np.cos(angll_rad), -9999. )


##############################################
coords = dict( 
    lon = ( ["lon"], Topo.lon.values ),
    lat  = ( ["lat"], Topo.lat.values  ), )

Xout = xr.Dataset( coords=coords  )


dims = ('lat','lon' )
Dar = xr.DataArray( data=mxdis, 
                    dims=dims,
                    attrs = Topo.MXDIS.attrs , )
Xout[ 'Ridge_Height' ]= Dar

dims = ('lat','lon' )
Dar = xr.DataArray( data=sin_angll_rad, 
                    dims=dims,
                    attrs = {'long_name': ' Sine of Ridge orientation angle (0=north-south ridge, 90=east-west ridge etc  )   ',
                             'units': '-1,1',}
                   , )
Xout[ 'Ridge_SinAngle' ]= Dar

dims = ('lat','lon' )
Dar = xr.DataArray( data=cos_angll_rad, 
                    dims=dims,
                    attrs = {'long_name': ' Cosine of Ridge orientation angle (0=north-south ridge, 90=east-west ridge etc  )  ',
                             'units': '-1,1',}
                   , )
Xout[ 'Ridge_CosAngle' ]= Dar

dims = ('lat','lon' )
Dar = xr.DataArray( data=isovar, 
                    dims=dims,
                    attrs = {'long_name': 'Residual non-ridge obstacle height',
                             'units': 'm',}
                   , )
Xout[ 'NonRidge_Resid' ]= Dar


dims = ('lat','lon' )
Dar = xr.DataArray( data=htopo, 
                    dims=dims,
                    attrs = {'long_name': ' grid mean topo elevation ',
                             'units': 'm',}
                   , )
Xout[ 'MeanTopo' ]= Dar





In [3]:
outfile='/glade/work/juliob/Topo/NCARTopoJTB/Topodata4ML.nc'

Xout.to_netcdf( outfile )